In [ ]:
!pip install transformers torchaudio librosa webrtcvad numpy torch uuid sentencepiece

In [1]:
import math
import uuid
from dataclasses import dataclass
from typing import List, Tuple

import librosa
import numpy as np
import torch
import webrtcvad


def _pcm16(audio: np.ndarray) -> bytes:
    audio_int16 = np.clip(audio * 32768.0, -32768, 32767).astype(np.int16)
    return audio_int16.tobytes()


def _frame_bytes(pcm_bytes: bytes, frame_size: int) -> List[bytes]:
    return [pcm_bytes[i : i + frame_size] for i in range(0, len(pcm_bytes), frame_size) if len(pcm_bytes[i : i + frame_size]) == frame_size]


@dataclass
class AudioChunk:
    chunk_id: str
    audio_path: str
    start_time: float
    end_time: float
    mel: np.ndarray
    duration: float
    waveform: torch.Tensor
    sample_rate: int


class AudioPreprocessor:
    """
    Minimal audio chunking + mel pipeline for model ingestion.
    Keeps mapping and frame bounds for 10-20s speech-only chunks.
    """

    def __init__(
        self,
        target_sr: int = 16000,
        min_chunk_s: float = 8.0,
        max_chunk_s: float = 20.0,
        vad_aggressiveness: int = 3,
        merge_gap_s: float = 0.1,
        drop_short_s: float = 0.5,
        pad_s: float = 0.1,
        overlap_frac: float = 0.05,
        mel_n_mels: int = 80,
        mel_win_ms: float = 25.0,
        mel_hop_ms: float = 10.0,
        max_mel_frames: int = 1000,
    ):
        self.sr = target_sr
        self.min_chunk_s = min_chunk_s
        self.max_chunk_s = max_chunk_s
        self.merge_gap_s = merge_gap_s
        self.drop_short_s = drop_short_s
        self.pad_s = pad_s
        self.overlap_frac = overlap_frac
        self.max_mel_frames = max_mel_frames
        self.hop_length = int(self.sr * mel_hop_ms / 1000)
        self.win_length = int(self.sr * mel_win_ms / 1000)
        self.n_mels = mel_n_mels
        self.vad = webrtcvad.Vad(vad_aggressiveness)

    def _detect_vad(self, audio: np.ndarray) -> List[Tuple[float, float]]:
        frame_duration_ms = 20  # WebRTC VAD supports 10/20/30ms
        frame_size = int(self.sr * frame_duration_ms / 1000)
        pcm = _pcm16(audio)
        frames = _frame_bytes(pcm, frame_size * 2)  # 2 bytes per sample
        intervals: List[Tuple[float, float]] = []
        start = None
        for idx, frame in enumerate(frames):
            ts = idx * frame_duration_ms / 1000.0
            has_speech = self.vad.is_speech(frame, self.sr)
            if has_speech and start is None:
                start = ts
            if not has_speech and start is not None:
                intervals.append((start, ts))
                start = None
        if start is not None:
            intervals.append((start, len(frames) * frame_duration_ms / 1000.0))
        merged: List[Tuple[float, float]] = []
        for s, e in intervals:
            if e - s < self.drop_short_s:
                continue
            if merged and s - merged[-1][1] <= self.merge_gap_s:
                merged[-1] = (merged[-1][0], e)
            else:
                merged.append((s, e))
        return merged

    def _segment_chunks(self, intervals: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
        chunks: List[Tuple[float, float]] = []
        if not intervals:
            return chunks
        cur_start, cur_end = intervals[0]
        for s, e in intervals[1:]:
            if (e - cur_start) <= self.max_chunk_s:
                cur_end = e
            else:
                chunks.append((cur_start, cur_end))
                cur_start, cur_end = s, e
        chunks.append((cur_start, cur_end))
        normalized: List[Tuple[float, float]] = []
        for seg in chunks:
            if normalized and (seg[1] - seg[0]) < self.min_chunk_s:
                prev_s, prev_e = normalized.pop()
                normalized.append((prev_s, seg[1]))
            else:
                normalized.append(seg)
        return normalized

    def _pad_and_overlap(self, chunks: List[Tuple[float, float]], audio_len_s: float) -> List[Tuple[float, float]]:
        padded: List[Tuple[float, float]] = []
        for i, (s, e) in enumerate(chunks):
            pad = self.pad_s
            s = max(0.0, s - pad)
            e = min(audio_len_s, e + pad)
            padded.append((s, e))
            if i < len(chunks) - 1:
                overlap = (e - s) * self.overlap_frac
                padded[-1] = (s, min(audio_len_s, e + overlap))
        return padded

    def _mel(self, audio: np.ndarray) -> np.ndarray:
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=self.sr,
            n_fft=self.win_length,
            hop_length=self.hop_length,
            win_length=self.win_length,
            n_mels=self.n_mels,
            power=2.0,
            center=True,
        )
        mel = librosa.power_to_db(mel).T  # [T, n_mels]
        return mel

    def _enforce_frame_limit(self, mel: np.ndarray) -> List[np.ndarray]:
        if mel.shape[0] <= self.max_mel_frames:
            return [mel]
        n_splits = math.ceil(mel.shape[0] / self.max_mel_frames)
        return np.array_split(mel, n_splits)

    def process(self, audio_path: str) -> List[AudioChunk]:
        audio, _ = librosa.load(audio_path, sr=self.sr, mono=True)
        audio = np.clip(audio / (np.max(np.abs(audio)) + 1e-9), -1.0, 1.0)
        vad_segments = self._detect_vad(audio)
        speech_chunks = self._segment_chunks(vad_segments)
        padded_chunks = self._pad_and_overlap(speech_chunks, len(audio) / self.sr)

        results: List[AudioChunk] = []
        for seg_start, seg_end in padded_chunks:
            start_idx = int(seg_start * self.sr)
            end_idx = int(seg_end * self.sr)
            clip = audio[start_idx:end_idx]
            if clip.size == 0:
                continue
            mel = self._mel(clip)
            sub_mels = self._enforce_frame_limit(mel)
            frame_dt = self.hop_length / self.sr
            for part_idx, sub in enumerate(sub_mels):
                offset_frames = part_idx * self.max_mel_frames
                sub_start = seg_start + offset_frames * frame_dt
                sub_end = sub_start + sub.shape[0] * frame_dt
                rel_start = int(offset_frames * self.hop_length)
                rel_end = rel_start + sub.shape[0] * self.hop_length
                rel_end = min(rel_end, clip.shape[0])
                wave_slice_np = clip[rel_start:rel_end]
                if wave_slice_np.size == 0:
                    continue
                waveform = torch.from_numpy(wave_slice_np).float()
                results.append(
                    AudioChunk(
                        chunk_id=str(uuid.uuid4()),
                        audio_path=audio_path,
                        start_time=sub_start,
                        end_time=sub_end,
                        mel=sub,
                        duration=sub.shape[0] * frame_dt,
                        waveform=waveform,
                        sample_rate=self.sr,
                    )
                )
        return results


if __name__ == "__main__":
    ap = AudioPreprocessor()
    chunks = ap.process("/mnt/devdrive/AIMS/SIH25/code/dataset/ps6_01_214.wav")
    print(chunks[0].mel.shape, chunks[0].start_time, chunks[0].end_time)
    for chunk in chunks:
        print(f"Chunk ID: {chunk.chunk_id}, Start: {chunk.start_time:.2f}s, End: {chunk.end_time:.2f}s, Duration: {chunk.duration:.2f}s, Mel shape: {chunk.mel.shape}")


/mnt/devdrive/AIMS/SIH25/code/venv/lib64/python3.13/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


(989, 80) 0.0 9.89
Chunk ID: 65506dd3-583a-4833-b13d-6b3f16600bf4, Start: 0.00s, End: 9.89s, Duration: 9.89s, Mel shape: (989, 80)
Chunk ID: f3d0f014-bafe-476f-93db-7c62cb5decdf, Start: 10.00s, End: 19.88s, Duration: 9.88s, Mel shape: (988, 80)
Chunk ID: 8bab7485-664a-476d-867a-3550cd3ee35d, Start: 19.20s, End: 26.16s, Duration: 6.96s, Mel shape: (696, 80)
Chunk ID: df93081a-68c2-4d00-8b49-34800e8f9948, Start: 29.20s, End: 36.16s, Duration: 6.96s, Mel shape: (696, 80)
Chunk ID: d14a1d47-b19b-472c-9cb9-d8b92e077fa2, Start: 39.20s, End: 46.16s, Duration: 6.96s, Mel shape: (696, 80)
Chunk ID: cf874703-4249-4dc6-831e-c617298bc604, Start: 40.88s, End: 47.93s, Duration: 7.05s, Mel shape: (705, 80)
Chunk ID: e0428b97-04fb-41fb-8388-a8d058d19be0, Start: 50.88s, End: 57.92s, Duration: 7.04s, Mel shape: (704, 80)
Chunk ID: 74b45d3a-3212-4b94-abf7-816a81b0f7da, Start: 60.88s, End: 67.92s, Duration: 7.04s, Mel shape: (704, 80)
Chunk ID: 98f48d73-73cd-4e93-9609-72ee8f20c423, Start: 61.38s, End: 70.

In [2]:
import torch
from transformers import Wav2Vec2Model

# Load frozen wav2vec2 XLSR
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large-xlsr-53")
for p in wav2vec.parameters():
    p.requires_grad = False
wav2vec.eval()


def extract_hidden_states(chunk_list, wav2vec_model: Wav2Vec2Model):
    """Pad waveforms, build mask, and return frozen encoder states."""
    device = next(wav2vec_model.parameters()).device
    wavs = [c.waveform.to(device).float().squeeze() for c in chunk_list]
    lengths = [w.shape[-1] for w in wavs if w.numel() > 0]
    if not lengths:
        return torch.empty(0, 0, 0, device=device), []

    max_len = max(lengths)
    padded = []
    for w in wavs:
        pad_len = max_len - w.shape[-1]
        if pad_len > 0:
            w = torch.nn.functional.pad(w, (0, pad_len))
        padded.append(w)

    batch = torch.stack(padded, dim=0)  # [B, T]
    mask = torch.zeros(batch.size(0), batch.size(1), dtype=torch.long, device=device)
    for i, L in enumerate(lengths):
        mask[i, :L] = 1

    with torch.no_grad():
        outputs = wav2vec_model(batch, attention_mask=mask)
        hidden = outputs.last_hidden_state  # [B, T_enc, 1024]

    return hidden, lengths


# if __name__ == "__main__":
#     ap = AudioPreprocessor()
#     chunks = ap.process("/mnt/devdrive/AIMS/SIH25/code/dataset/ps6_01_214.wav")
#     hidden, lengths = extract_hidden_states(chunks, wav2vec)
#     print(hidden.shape, lengths)
#     pass


/mnt/devdrive/AIMS/SIH25/code/venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
import torch.nn as nn


class AudioProjection(nn.Module):
    def __init__(self, in_dim=1024, out_dim=1024):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.ln = nn.LayerNorm(out_dim)

    def forward(self, audio_h):
        return self.ln(self.proj(audio_h))


class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.base = base_linear
        self.r = r
        self.alpha = alpha
        if r > 0:
            self.A = nn.Parameter(torch.randn(base_linear.in_features, r) * 0.01)
            self.B = nn.Parameter(torch.zeros(r, base_linear.out_features))
            self.scaling = alpha / r
        else:
            self.register_parameter("A", None)
            self.register_parameter("B", None)
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        base_out = self.base(x)
        if self.r == 0:
            return base_out
        lora_out = (x @ self.A @ self.B) * self.scaling
        return base_out + lora_out


class AudioCrossAttention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, r=8):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_Q = LoRALinear(nn.Linear(d_model, d_model), r=r)
        self.W_K = LoRALinear(nn.Linear(d_model, d_model), r=r)
        self.W_V = LoRALinear(nn.Linear(d_model, d_model), r=r)
        self.W_O = LoRALinear(nn.Linear(d_model, d_model), r=r)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, text_h, audio_h):
        B, T_q, _ = text_h.shape
        _, T_a, _ = audio_h.shape
        q = self.W_Q(text_h)
        k = self.W_K(audio_h)
        v = self.W_V(audio_h)
        q = q.reshape(B, T_q, self.n_heads, self.d_head).transpose(1, 2)
        k = k.reshape(B, T_a, self.n_heads, self.d_head).transpose(1, 2)
        v = v.reshape(B, T_a, self.n_heads, self.d_head).transpose(1, 2)
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) / (self.d_head**0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        ctx = torch.matmul(attn_weights, v)
        ctx = ctx.transpose(1, 2).reshape(B, T_q, self.d_model)
        return self.ln(self.W_O(ctx))


class ResidualAdapter(nn.Module):
    def __init__(self, d_model=1024, bottleneck=256):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, d_model)
        self.ln = nn.LayerNorm(d_model)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.ln(x + self.up(self.act(self.down(x))))


class GatedFusion(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.gate = nn.Linear(d_model * 2, d_model)

    def forward(self, text_h, audio_ctx):
        g = torch.sigmoid(self.gate(torch.cat([text_h, audio_ctx], dim=-1)))
        return text_h * (1 - g) + audio_ctx * g


class AudioFusionBlock(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, r=8, bottleneck=256):
        super().__init__()
        self.cross_attn = AudioCrossAttention(d_model, n_heads, r)
        self.adapter = ResidualAdapter(d_model, bottleneck)
        self.gate = GatedFusion(d_model)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, text_h, audio_h):
        audio_ctx = self.cross_attn(text_h, audio_h)
        fused = self.gate(text_h, audio_ctx)
        return self.adapter(fused)


# Example wiring (pseudo):
# audio_h = AudioProjection(out_dim=t5.config.d_model)(hidden_audio)   # [B, T_a, d_model]
# text_h = decoder_layer(text_h)                                       # frozen
# text_h = fusion_block(text_h, audio_h)                               # trainable adapters

In [4]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, BitsAndBytesConfig
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F


class AudioFFNProjector(nn.Module):
    def __init__(self, in_dim=1024, d_model=1024, hidden=2048):
        super().__init__()
        self.w1 = nn.Linear(in_dim, hidden)
        self.w2 = nn.Linear(hidden, d_model)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.ln(F.gelu(self.w2(F.gelu(self.w1(x)))))


def freeze_module(mod):
    for p in mod.parameters():
        p.requires_grad = False
    return mod


def apply_lora_to_decoder_self_attn(t5_model, num_top_layers: int = 4, r: int = 8, alpha: int = 16):
    """Wrap top decoder self-attn projections with LoRA; base weights stay frozen, LoRA is trainable."""
    total_layers = len(t5_model.decoder.block)
    start = max(0, total_layers - num_top_layers)
    for i in range(start, total_layers):
        sa = t5_model.decoder.block[i].layer[0].SelfAttention
        sa.q = LoRALinear(sa.q, r=r, alpha=alpha)
        sa.k = LoRALinear(sa.k, r=r, alpha=alpha)
        sa.v = LoRALinear(sa.v, r=r, alpha=alpha)
        sa.o = LoRALinear(sa.o, r=r, alpha=alpha)
    return t5_model


def cast_lora_params(module: nn.Module, dtype):
    """Cast only trainable LoRA A/B parameters to dtype, keep frozen base weights as-is."""
    for m in module.modules():
        if isinstance(m, LoRALinear):
            if m.A is not None:
                m.A.data = m.A.data.to(dtype)
            if m.B is not None:
                m.B.data = m.B.data.to(dtype)


def build_bnb_int8_config():
    return BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_skip_modules=None,
        llm_int8_has_fp16_weight=False,
        bnb_8bit_compute_dtype=torch.float16,
    )


tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")


def fuse_text_and_audio(
    question: str,
    audio_path: str,
    preproc: AudioPreprocessor,
    wav2vec_model: Wav2Vec2Model,
    t5_model: T5ForConditionalGeneration,
    projector: AudioFFNProjector,
    fusion_block: AudioFusionBlock,
    device=None,
):
    """Fusion helper when a t5 instance is provided by the caller (no new t5 instantiation)."""
    device = device or next(t5_model.parameters()).device
    # Text encoding (frozen encoder)
    text_inputs = tokenizer(question, return_tensors="pt").to(device)
    enc_out = t5_model.encoder(**text_inputs)
    text_h = enc_out.last_hidden_state  # [B, T_q, d_model]
    text_mask = text_inputs.get("attention_mask")

    # Audio to wav2vec2 hidden -> projection
    chunks = preproc.process(audio_path)
    hidden_audio, _ = extract_hidden_states(chunks, wav2vec_model)
    hidden_audio = hidden_audio.to(device)  # [B, T_a, 1024]
    audio_proj = projector(hidden_audio)    # [B, T_a, d_model]

    # Cross-attn fusion (trainable adapters only)
    fusion_block = fusion_block.to(device)
    fused_text = fusion_block(text_h, audio_proj)  # [B, T_q, d_model]

    fused_enc = BaseModelOutput(last_hidden_state=fused_text)

    # Minimal decoder call (teacher-forcing with start token)
    start_id = t5_model.config.decoder_start_token_id or t5_model.config.pad_token_id
    dec_in = torch.tensor([[start_id]], device=device)
    outputs = t5_model(
        encoder_outputs=fused_enc,
        decoder_input_ids=dec_in,
        decoder_attention_mask=text_mask,
    )
    return outputs.logits, fused_text

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [5]:
class AudioQAPipeline(nn.Module):
    """End-to-end frozen wav2vec2 + frozen FLAN-T5 + trainable fusion adapters (int8 via bitsandbytes)."""

    def __init__(
        self,
        preproc: AudioPreprocessor,
        wav2vec_model: Wav2Vec2Model,
        lora_top_layers: int = 4,
        lora_r: int = 8,
        lora_alpha: int = 16,
        device: str | None = None,
    ):
        super().__init__()
        if not torch.cuda.is_available():
            raise RuntimeError("bitsandbytes int8 path requires CUDA GPU.")
        self.device = torch.device(device or "cuda")
        self.preproc = preproc
        self.wav2vec = wav2vec_model.to(self.device)
        freeze_module(self.wav2vec)
        self.tokenizer = tokenizer
        bnb_config = build_bnb_int8_config()
        # Instantiate T5 only here, load quantized with bitsandbytes int8, freeze base params, then add trainable LoRA.
        self.t5 = T5ForConditionalGeneration.from_pretrained(
            "google/flan-t5-large",
            quantization_config=bnb_config,
            device_map={"": self.device},
        )
        freeze_module(self.t5)
        apply_lora_to_decoder_self_attn(self.t5, num_top_layers=lora_top_layers, r=lora_r, alpha=lora_alpha)
        self.model_dtype = torch.float16
        cast_lora_params(self.t5, self.model_dtype)
        cfg = self.t5.config
        self.projector = AudioFFNProjector(in_dim=1024, d_model=cfg.d_model, hidden=2 * cfg.d_model).to(self.device, dtype=self.model_dtype)
        self.fusion_block = AudioFusionBlock(d_model=cfg.d_model, n_heads=cfg.num_heads, r=lora_r, bottleneck=256).to(self.device, dtype=self.model_dtype)
        cast_lora_params(self.fusion_block, self.model_dtype)

    def forward(self, question: str, audio_path: str, device=None, max_new_tokens: int = 32):
        # Allow override but default to pipeline device
        device = torch.device(device) if device else self.device
        # Text encode
        text_inputs = self.tokenizer(question, return_tensors="pt").to(device)
        enc_out = self.t5.encoder(**text_inputs)
        text_h = enc_out.last_hidden_state.to(self.model_dtype)  # [B_text, T_q, d_model]
        text_mask = text_inputs.get("attention_mask")

        # Audio encode
        chunks = self.preproc.process(audio_path)
        hidden_audio, _ = extract_hidden_states(chunks, self.wav2vec)
        hidden_audio = hidden_audio.to(device, dtype=self.model_dtype)
        audio_proj = self.projector(hidden_audio)  # [B_audio, T_a, d_model]

        # Align batch dims: collapse audio batch to single batch to match text batch (B=1)
        if audio_proj.dim() == 3 and audio_proj.size(0) != text_h.size(0):
            audio_proj = audio_proj.reshape(1, -1, audio_proj.size(-1))

        # Fusion
        fused_text = self.fusion_block(text_h, audio_proj)
        fused_enc = BaseModelOutput(last_hidden_state=fused_text)

        # Autoregressive decode (greedy for brevity)
        start_id = self.t5.config.decoder_start_token_id or self.t5.config.pad_token_id
        dec_ids = torch.tensor([[start_id]], device=device)
        for _ in range(max_new_tokens):
            out = self.t5(
                encoder_outputs=fused_enc,
                decoder_input_ids=dec_ids,
                decoder_attention_mask=text_mask,
            )
            next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            dec_ids = torch.cat([dec_ids, next_id], dim=1)
            if next_id.item() == self.tokenizer.eos_token_id:
                break
        return dec_ids


In [ ]:
# Usage (pseudo):
ap = AudioPreprocessor()
pipe = AudioQAPipeline(ap, wav2vec)
pipe.to("cuda")
ans_ids = pipe("What is said?", "../dataset/ID01.wav")
print(pipe.tokenizer.decode(ans_ids[0], skip_special_tokens=True))